# Trakπ — inline test against a locally-orchestrated OTP

One test run driven entirely from the library, with **every SPI implementation inline** in this
notebook: the orchestrator that boots a local OTP, the planner that queries it, the request source,
the KPIs and the results writer. No CLI, no config files, no `reference/otp`.

The orchestrator follows the `PlannerOrchestrator` SPI (`start` / `stop` / `status`). `start` is
**eager** — it launches OTP and returns immediately — and you poll `status` for readiness by
re-running the poll cell until it reports `READY`.

Requires the Kotlin Jupyter kernel, a local `mvn -f core/pom.xml install`, a built OTP shaded jar,
and a prebuilt OTP graph.

## 1. Dependencies

Point `@file:DependsOn` straight at the jars `mvn -f core/pom.xml install` built — this skips Maven
resolution (and the SNAPSHOT-from-mavenLocal problems that come with it). Adjust the paths if your
trakpi checkout is elsewhere. `kotlinx-serialization` comes from the project classpath (Kotlin
Notebook setting: *All Project Libraries*); if you don't have that on, add a Maven-coordinate
`@file:DependsOn` for it.

In [1]:
@file:DependsOn("/workspace/trakpi/core/common/target/common-1.0.0-SNAPSHOT.jar")
@file:DependsOn("/workspace/trakpi/core/tester/target/tester-1.0.0-SNAPSHOT.jar")
@file:DependsOn("/workspace/trakpi/core/orchestrator/target/orchestrator-1.0.0-SNAPSHOT.jar")

In [2]:
import java.net.URI
import java.net.http.HttpClient
import java.net.http.HttpRequest
import java.net.http.HttpResponse.BodyHandlers
import java.io.File
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.StandardCopyOption
import java.time.Duration
import java.time.Instant
import java.util.concurrent.TimeUnit
import java.util.zip.ZipInputStream
import kotlinx.serialization.json.*

import org.opentripplanner.trakpi.common.PlannerVersion
import org.opentripplanner.trakpi.common.TestsetVersion
import org.opentripplanner.trakpi.orchestrator.PlannerOrchestrator
import org.opentripplanner.trakpi.orchestrator.PlannerStatus
import org.opentripplanner.trakpi.tester.Tester
import org.opentripplanner.trakpi.tester.spi.*
import org.opentripplanner.trakpi.tester.spi.kpi.*

## 2. The planner

A `TravelPlanner` that POSTs a GraphQL query to an OTP Transmodel endpoint. The endpoint is supplied
at construction, so the same planner works against the dev API or a local OTP — here it's pointed at
the locally-orchestrated server.

In [3]:
val CLIENT_NAME = "entur-trakpi-notebook"

data class GraphQlRequest(val query: String) : TravelPlannerRequest

class OtpPlanner(private val endpoint: String, private val clientName: String) : TravelPlanner<GraphQlRequest> {
    // HTTP/1.1 explicitly: OTP's Grizzly server can truncate a POST body over the default h2c upgrade.
    private val http = HttpClient.newBuilder()
        .version(HttpClient.Version.HTTP_1_1).connectTimeout(Duration.ofSeconds(10)).build()

    override fun execute(request: GraphQlRequest): TravelPlannerResponse {
        val body = buildJsonObject { put("query", JsonPrimitive(request.query)) }.toString()
        val httpRequest = HttpRequest.newBuilder()
            .uri(URI.create(endpoint))
            .header("Content-Type", "application/json")
            .header("ET-Client-Name", clientName)
            .timeout(Duration.ofSeconds(30))
            .POST(HttpRequest.BodyPublishers.ofString(body))
            .build()
        val res = http.send(httpRequest, BodyHandlers.ofString())
        val json = runCatching { Json.parseToJsonElement(res.body()).jsonObject }.getOrNull()
        val data = json?.get("data") as? JsonObject
        val errors = json?.get("errors") as? JsonArray
        val success = res.statusCode() / 100 == 2 && data != null && errors.isNullOrEmpty()
        return TravelPlannerResponse(
            raw = res.body(),
            success = success,
            method = data?.keys?.firstOrNull() ?: "unknown",
            attributes = mapOf("http_status_code" to res.statusCode().toString()),
        )
    }
}

## 3. The requests

Literal `trip` queries, sourced by an inline `RequestFileLoader` and parsed by an inline
`RequestLoader`. These are **intra-Oslo** trips on purpose: the demo graph below covers Oslo only,
so cross-country requests would come back empty. Each query selects just the fields the KPIs need.

In [4]:
fun tripQuery(fromLat: Double, fromLon: Double, toLat: Double, toLon: Double) = """
    {
      trip(
        from: { coordinates: { latitude: $fromLat, longitude: $fromLon } }
        to:   { coordinates: { latitude: $toLat,  longitude: $toLon } }
      ) {
        tripPatterns { duration legs { mode } }
        debugOutput { totalTime }
      }
    }
""".trimIndent()

// id -> raw request body. Oslo S, Nationaltheatret, Majorstuen, Blindern.
val requests = mapOf(
    "oslos-nationalth" to tripQuery(59.911, 10.750, 59.914, 10.733),
    "oslos-majorstuen" to tripQuery(59.911, 10.750, 59.929, 10.714),
    "oslos-blindern"   to tripQuery(59.911, 10.750, 59.939, 10.721),
)

class InlineRequestFileLoader(private val bodies: Map<String, String>) : RequestFileLoader {
    override fun loadAll(testsetVersion: TestsetVersion): List<RequestFile> =
        bodies.map { (id, body) -> RequestFile(id, body) }
}

class GraphQlRequestLoader : RequestLoader<GraphQlRequest> {
    override fun load(file: RequestFile): GraphQlRequest = GraphQlRequest(file.body)
}

## 4. The KPIs

Each `KPICalculator` reads one number from a response, or returns `null` when it doesn't apply.

In [5]:
fun TravelPlannerResponse.tripObject(): JsonObject? =
    runCatching { Json.parseToJsonElement(raw).jsonObject["data"]?.jsonObject?.get("trip")?.jsonObject }.getOrNull()

class ItineraryCount : KPICalculator {
    override fun calculate(response: TravelPlannerResponse): Kpi? {
        val trip = response.tripObject() ?: return null
        val patterns = (trip["tripPatterns"] as? JsonArray)?.size ?: 0
        return Kpi("itineraryCount", patterns.toDouble())
    }
}

class RoutingTimeMs : KPICalculator {
    override fun calculate(response: TravelPlannerResponse): Kpi? {
        val nanos = response.tripObject()?.get("debugOutput")?.jsonObject
            ?.get("totalTime")?.jsonPrimitive?.longOrNull ?: return null
        return Kpi("routingTimeMs", nanos / 1_000_000.0)
    }
}

class SuccessKpi : KPICalculator {
    override fun calculate(response: TravelPlannerResponse) =
        Kpi("success", if (response.success) 1.0 else 0.0)
}

## 5. The results writer

Collects results in memory so the notebook can render them.

In [6]:
val collected = mutableListOf<TestCaseResult>()

class InMemoryResultsWriter : ResultsWriter {
    override fun store(run: RunMetadata, result: TestCaseResult) { collected += result }
}

## 6. The orchestrator

An inline `PlannerOrchestrator` that runs OTP as a local subprocess. `start` launches
`java -jar <otp> --load <graph>` and returns at once; `status` reports `NOT_RUNNING` / `STARTING` /
`READY` / `FAILED` by asking OTP for its `serverInfo`; `stop` kills the process. `awaitReady` polls
`status` a bounded number of times so the poll cell returns promptly and you re-run it until `READY`.

In [7]:
// --- data preparation helpers (pure Kotlin; no shell) ---
val NETEX_BASE_URL = "https://storage.googleapis.com/marduk-production/outbound/netex"
val NORWAY_NETEX: Path = Path.of("/workspace/otp-data/otp/norway/netex")
val RUT_VIEW: Path = Path.of("/workspace/motis-run/input/rut-netex")

// Launch a planner build/import as a subprocess; output appended to prepare.log.
fun run(vararg cmd: String, dir: File? = null) {
    val p = ProcessBuilder(*cmd).directory(dir).redirectErrorStream(true)
        .redirectOutput(ProcessBuilder.Redirect.appendTo(File("prepare.log"))).start()
    check(p.waitFor() == 0) { "command failed: ${cmd.joinToString(" ")} - see prepare.log" }
}

fun httpDownload(url: String, dest: Path) {
    val http = HttpClient.newBuilder().followRedirects(HttpClient.Redirect.NORMAL).connectTimeout(Duration.ofSeconds(30)).build()
    val res = http.send(HttpRequest.newBuilder(URI.create(url)).GET().build(), BodyHandlers.ofInputStream())
    check(res.statusCode() / 100 == 2) { "download failed ($url): HTTP ${res.statusCode()}" }
    res.body().use { Files.copy(it, dest, StandardCopyOption.REPLACE_EXISTING) }
}

// Extract every entry flat (by file name) into destDir.
fun unzipFlat(zip: Path, destDir: Path) {
    ZipInputStream(Files.newInputStream(zip)).use { zis ->
        var e = zis.nextEntry
        while (e != null) {
            if (!e.isDirectory) Files.copy(zis, destDir.resolve(Path.of(e.name).fileName), StandardCopyOption.REPLACE_EXISTING)
            e = zis.nextEntry
        }
    }
}

fun Path.entries(): List<Path> = Files.newDirectoryStream(this).use { it.toList() }

// Download one codespace fresh from Entur into norway/netex (a single codespace, not full Norway).
fun downloadNetex(codespace: String) {
    val cs = codespace.lowercase(); val CS = codespace.uppercase()
    Files.createDirectories(NORWAY_NETEX)
    val zip = NORWAY_NETEX.resolve("_$cs.zip")
    println("[prepare] download $CS netex from Entur")
    httpDownload("$NETEX_BASE_URL/rb_$cs-aggregated-netex.zip", zip)
    NORWAY_NETEX.entries().filter { val n = it.fileName.toString(); n.startsWith("${CS}_") || n.startsWith("_${CS}_") }.forEach { Files.deleteIfExists(it) }
    unzipFlat(zip, NORWAY_NETEX)
    Files.deleteIfExists(zip)
}

// Rebuild the RUT-only view (line + shared files + the stable stops file) both engines read.
fun refreshRutView() {
    Files.createDirectories(RUT_VIEW)
    val keep = { n: String -> n.startsWith("RUT_") || n.startsWith("_RUT_") || n == "_stops.xml" }
    RUT_VIEW.entries().filter { keep(it.fileName.toString()) }.forEach { Files.deleteIfExists(it) }
    NORWAY_NETEX.entries().filter { keep(it.fileName.toString()) }.forEach { Files.createSymbolicLink(RUT_VIEW.resolve(it.fileName), it) }
}

// Inline readiness probe: OTP answers `serverInfo` once it is serving.
fun otpResponds(endpoint: String, clientName: String): Boolean = try {
    val http = HttpClient.newBuilder().version(HttpClient.Version.HTTP_1_1)
        .connectTimeout(Duration.ofSeconds(5)).build()
    val req = HttpRequest.newBuilder().uri(URI.create(endpoint))
        .header("Content-Type", "application/json").header("ET-Client-Name", clientName)
        .timeout(Duration.ofSeconds(10))
        .POST(HttpRequest.BodyPublishers.ofString("""{"query":"{ serverInfo { version } }"}""")).build()
    val res = http.send(req, BodyHandlers.ofString())
    val data = Json.parseToJsonElement(res.body()).jsonObject["data"] as? JsonObject
    res.statusCode() / 100 == 2 && data?.get("serverInfo") != null
} catch (e: Exception) { false }

class LocalOtpOrchestrator(
    val otpJar: String,
    val graphDir: String,
    val port: Int = 8084,
    val clientName: String = CLIENT_NAME,
    val heap: String = "-Xmx4g",
    val logFile: String = "otp-server.log",
) : PlannerOrchestrator {
    val endpoint = "http://localhost:$port/otp/transmodel/v3"
    private var process: Process? = null

    // prepare(): fetch fresh RUT timetable, refresh the shared view, then build the graph.
    override fun prepare(version: PlannerVersion, args: String?) {
        downloadNetex("RUT")
        refreshRutView()
        println("[prepare] building graph (slow) -> $graphDir/graph.obj")
        run("java", heap, "-jar", otpJar, "--build", "--save", graphDir)
        println("[prepare] graph built")
    }

    override fun start(version: PlannerVersion, args: String?) {
        // Reuse an instance already serving on this port (e.g. one left running by an earlier cell
        // run) instead of launching a clashing one. Stop/kill it first to force a fresh start.
        if (otpResponds(endpoint, clientName)) { println("[local-otp] $endpoint already serving - reusing it"); return }
        check(process == null) { "OTP already launched - stop() first" }
        println("[local-otp] launching $otpJar --load $graphDir --port $port  (logs -> $logFile)")
        process = ProcessBuilder("java", heap, "-jar", otpJar, "--load", graphDir, "--port", "$port")
            .redirectErrorStream(true)
            .redirectOutput(ProcessBuilder.Redirect.appendTo(File(logFile)))
            .start()
    }

    override fun stop(version: PlannerVersion) {
        process?.let {
            println("[local-otp] stopping")
            it.destroy()
            if (!it.waitFor(20, TimeUnit.SECONDS)) it.destroyForcibly()
        }
        process = null
    }

    override fun status(version: PlannerVersion): PlannerStatus {
        val p = process
        return when {
            otpResponds(endpoint, clientName) -> PlannerStatus.READY   // launched or reused
            p != null && !p.isAlive -> PlannerStatus.FAILED
            p != null -> PlannerStatus.STARTING
            else -> PlannerStatus.NOT_RUNNING
        }
    }

    /** Poll status up to [attempts] times, [interval] apart; return as soon as it settles. */
    fun awaitReady(version: PlannerVersion, attempts: Int = 10, interval: Duration = Duration.ofSeconds(3)): PlannerStatus {
        repeat(attempts) {
            when (val s = status(version)) {
                PlannerStatus.READY, PlannerStatus.FAILED, PlannerStatus.NOT_RUNNING -> return s
                PlannerStatus.STARTING -> Thread.sleep(interval.toMillis())
            }
        }
        return status(version)
    }
}

## 7. Prepare — download fresh NeTEx, then build the graph

`prepare()` is the SPI's data step: it downloads a fresh single-codespace (RUT/Oslo) NeTEx from
Entur, refreshes the shared view, and builds `graph.obj` with the same jar it'll serve from. Fresh
data matters — a stale timetable whose validity has expired builds a graph with **zero trips**. It's
slow and wants a big heap; run once, then comment it out. Watch `prepare.log`.

A `graph.obj` only loads in an OTP whose **serialization id matches** the jar that built it, so never
reuse a prebuilt graph from a different build.

In [8]:
val orchestrator = LocalOtpOrchestrator(otpJar = "/workspace/otp/otp-shaded/target/otp-shaded-2.10.0-SNAPSHOT.jar", graphDir = "/workspace/otp-data/otp/oslo-shared")
val version = PlannerVersion("local")

// Run once (or whenever you change the OTP jar / want fresh data), then comment out:
orchestrator.prepare(version, null)

[prepare] download RUT netex from Entur
[prepare] building graph (slow) -> /workspace/otp-data/otp/oslo-shared/graph.obj
[prepare] graph built


## 8. Start OTP (returns immediately)

Launches the server and returns at once. Loading the graph takes tens of seconds — poll for
readiness in the next cell.

In [16]:
orchestrator.start(version, null)

[local-otp] launching /workspace/otp/otp-shaded/target/otp-shaded-2.10.0-SNAPSHOT.jar --load /workspace/otp-data/otp/oslo-shared --port 8084  (logs -> otp-server.log)


## 9. Poll until ready

**Re-run this cell** while it returns `STARTING`. `READY` means go; `FAILED` means the process died —
read `otp-server.log` in the notebook's working directory.

In [17]:
orchestrator.awaitReady(version)

READY

## 10. Run the test

Only once the poll cell says `READY`. You can re-run this against the same running server as often as
you like — the graph-load cost is paid once.

In [18]:
check(orchestrator.status(version) == PlannerStatus.READY) { "OTP is not READY yet - re-run the poll cell" }
collected.clear()
Tester(
    run = RunMetadata.create(
        version = version, application = "otp", startedAt = Instant.now(),
        testsetVersion = TestsetVersion("inline"), origin = "notebook",
    ),
    requestFileLoader = InlineRequestFileLoader(requests),
    requestLoader = GraphQlRequestLoader(),
    travelPlanner = OtpPlanner(orchestrator.endpoint, CLIENT_NAME),
    kpiCalculators = listOf(ItineraryCount(), RoutingTimeMs(), SuccessKpi()),
    resultsWriter = InMemoryResultsWriter(),
).run()
"Ran ${collected.size} request(s) against ${orchestrator.endpoint}"

100% complete. 3/3 requests processed.


Ran 3 request(s) against http://localhost:8084/otp/transmodel/v3

## 11. Results

In [19]:
val kpiNames = collected.flatMap { it.kpis.map { k -> k.name } }.distinct().sorted()
val header = "<tr><th>request</th><th>ok</th>" + kpiNames.joinToString("") { "<th>$it</th>" } + "</tr>"
val rows = collected.joinToString("") { r ->
    val byName = r.kpis.associate { it.name to it.value }
    val cells = kpiNames.joinToString("") { name ->
        val v = byName[name]
        val text = when {
            v == null -> "-"
            v == v.toLong().toDouble() -> v.toLong().toString()
            else -> "%.1f".format(v)
        }
        "<td>$text</td>"
    }
    "<tr><td>${r.requestId}</td><td>${if (r.success) "OK" else "FAIL"}</td>$cells</tr>"
}
HTML("<table>$header$rows</table>")

request,ok,itineraryCount,routingTimeMs,success
oslos-nationalth,OK,20,289.6,1
oslos-majorstuen,OK,50,160.2,1
oslos-blindern,OK,17,82.6,1


## 12. Stop OTP

Run when you're done. Leaving OTP up between tests is fine (and cheaper); this frees the port and
the process.

In [13]:
orchestrator.stop(version)

[local-otp] stopping
